In [ ]:
import pandas as pd
import numpy as np

# Load raw data (ELT best practice)
df = pd.read_csv('data/raw/churn_raw.csv')

df.head()

In [ ]:
# Shape check
print("Initial Shape:", df.shape)

# Duplicate check
duplicates = df.duplicated().sum()
print("Duplicate rows:", duplicates)

# Drop duplicates
df = df.drop_duplicates()

print("Shape after removing duplicates:", df.shape)

In [ ]:
# Drop irrelevant columns
df.drop(['customer_id', 'branch_code', 'city'], axis=1, inplace=True)

df.head()

In [ ]:
# Gender → MODE (most frequent category)
df['gender'].fillna(df['gender'].mode()[0], inplace=True)

# Dependents → MEDIAN (robust to outliers)
df['dependents'].fillna(df['dependents'].median(), inplace=True)

# Occupation → label missing explicitly
df['occupation'].fillna('Unknown', inplace=True)

# Clean gender
df['gender'] = df['gender'].replace('Other', df['gender'].mode()[0])

df.isnull().sum()

In [ ]:
# Monetary columns
money_cols = [
    'current_balance',
    'previous_month_end_balance',
    'average_monthly_balance_prevQ',
    'average_monthly_balance_prevQ2',
    'current_month_credit',
    'previous_month_credit',
    'current_month_debit',
    'previous_month_debit',
    'current_month_balance',
    'previous_month_balance'
]

# Replace negative values
for col in money_cols:
    df[col] = df[col].apply(lambda x: 0 if x < 0 else x)

# Cap outliers
for col in money_cols:
    lower = df[col].quantile(0.01)
    upper = df[col].quantile(0.99)
    df[col] = df[col].clip(lower, upper)

df[money_cols].describe()

In [ ]:
# Convert to datetime
df['last_transaction'] = pd.to_datetime(df['last_transaction'], errors='coerce')

# Extract features
df['transaction_year'] = df['last_transaction'].dt.year
df['transaction_month'] = df['last_transaction'].dt.month
df['transaction_day'] = df['last_transaction'].dt.day

# Missing flag
df['no_transaction_flag'] = df['last_transaction'].isna().astype(int)

df[['last_transaction', 'transaction_year', 'transaction_month']].head()

In [ ]:
# Age groups
df['age_group'] = pd.cut(
    df['age'],
    bins=[0, 25, 40, 60, 100],
    labels=['Young', 'Adult', 'Middle Age', 'Senior']
)

# Balance segments
df['balance_segment'] = pd.cut(
    df['current_balance'],
    bins=[-1, 1000, 10000, 100000, df['current_balance'].max()],
    labels=['Low', 'Medium', 'High', 'Very High']
)

# Total transactions
df['total_transactions'] = df['current_month_credit'] + df['current_month_debit']

# Activity level
df['activity_level'] = pd.cut(
    df['total_transactions'],
    bins=[-1, 1000, 10000, 50000, df['total_transactions'].max()],
    labels=['Low Activity', 'Moderate', 'High', 'Very High']
)

df[['age_group', 'balance_segment', 'activity_level']].head()

In [ ]:
# Final validation
print("Final Shape:", df.shape)

print("\nNegative values check:")
for col in money_cols:
    print(col, (df[col] < 0).sum())

In [ ]:
import os

os.makedirs('data/clean', exist_ok=True)

df.to_csv('data/clean/churn_clean.csv', index=False)

print("Clean dataset saved successfully!")